# Pre-processing pipeline data - datasetgeneratie voor unsupervised anomaliedetectie

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`  

## Doel

Deze notebook zet _exports_ van het GBS-platform (Johnson Controls) om naar _consistente_, _reproduceerbare_ **trainingsdata** voor een _unsupervised anomaliedetectiemodel_ voor hybride HVAC-systemen. De pipeline bevat `inlezen`, `opschonen`, `normaliseren`, `feature-engineering` en `export` naar een verwerkbaar bestandsformaat.

### Verwachte brondata

- Lijst met object-id's uit Johnson Controls die via de API de brondata op kan vragen. 
- Start- en einddatum in te lezen data.

De lijst ziet er als volgt uit:

```
id,func_datapunt_nr,datapunt_naam,object_id_gbs
1,44,temperatuur zone 1,11.02 DUNANT 1 - Kantoren links.Temperatuur,ba298cd6-2e20-5d5e-8b23-dd6f36801293
2,44,temperatuur zone 1,11.02 DUNANT 1 -Kantoren links.Temperatuur,b70bb261-c83d-5216-beed-0ef6ed5dc2fe
```

### Stappen

1. Inlezen van de csv met object-id's GBS
2. Data ophalen en opslaan via API uit GBS voor gevraagde tijdsperiode van alle id's
3. Opschonen en valideren van de opgehaalde data
4. Samenvoegen van geaggregeerde datapunten tot de functionele datapunten
5. Ophalen weerdata (KMI AWS 10 minuten) gevraagde punten
6. Datapunten synchroniseren tot 10-minuten intervallen en samenvoegen tot 1 dataset
7. Wegschrijven van de samengevoegde dataset

### Gebruik

- Invullen van CSV met object-id's voor het doelgebouw en plaatsen in de map `../data/bronnen.csv`
- Plaatsen van CSV met kmi aws data met daarin de gevraagde periode in `../data/raw/weerdata.csv`
- Doorlopen van de stappen en controleren van uitkomsten
- Bestanden komen terecht in `../data/pre-processed`

In [ ]:
import pandas as pd
import requests
import urllib3
from datetime import datetime, date, time, timedelta, timezone
from dotenv import load_dotenv
import os

## VARIABELEN

In [ ]:
# Tijdsvariabelen
START_DATUM = datetime(2026, 3, 9, 0, 0, 0, tzinfo=timezone.utc) # vul in jaar, datum, dag
EIND_DATUM  = datetime(2026, 4, 24, 23, 59, 59, tzinfo=timezone.utc)
GEBOUWNAAM = 'dunant1'

In [ ]:
# Bestanden
BRON_CSV = f'bronbestanden/bronnen-{GEBOUWNAAM}.csv'

In [ ]:
# Laad het .env bestand
load_dotenv('../.env', override=True)

## STAP 1 - inladen van brondata

In [ ]:
datapunten = pd.read_csv(BRON_CSV, index_col='id')
datapunten.head()

## STAP 2 - Data ophalen uit GBS

In [ ]:
# Schakel SSL-waarschuwingen uit (vaak nodig bij self-signed certificaten van Metasys servers)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
# --- Configuratie ---
base_url = os.getenv('BASE_URL')
username = os.getenv('USERNAME')
password = os.getenv('PASSWORD')

### Inloggen API JC

In [ ]:
# --- 1. Inloggen op de API ---
login_url = f"{base_url}/login"
login_response = requests.post(login_url, json={"username": username, "password": password}, verify=False)
login_response.raise_for_status()

access_token = login_response.json().get("accessToken")
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}
print("Succesvol ingelogd!")

### Ophaal functies

hieronder kan je de functies vinden waarmee de data opgehaald kan worden op de juiste manier.

In [ ]:
def haal_meetwaarden_op(object_id_gbs, start_dt, eind_dt, headers):
    """Haalt alle tijdreeks-data op voor één uniek Johnson Controls object."""
    # 1. URL van de samples ophalen
    trend_url = f"{base_url}/timeSeries/{object_id_gbs}/trendedAttributes"
    res = requests.get(trend_url, headers=headers, verify=False)
    
    if res.status_code != 200 or not res.json().get("items"):
        return []
        
    base_samples_url = res.json()["items"][0]["samplesUrl"].split('?')[0]
    
    # 2. Instellingen voor ophalen (zet datetime objecten om naar string)
    params = {
        "startSampleTime": start_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "endSampleTime": eind_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "pageSize": 1000,
        "page": 1,
        "sort": "sampleTime"
    }
    
    # 3. Paginatie: blijf data ophalen zolang er volgende pagina's zijn
    alle_samples = []
    while True:
        resp = requests.get(base_samples_url, headers=headers, params=params, verify=False)
        if resp.status_code != 200:
            break
            
        data = resp.json()
        alle_samples.extend(data.get("items", []))
        
        if not data.get("next"):
            break
        params["page"] += 1
        
    return alle_samples

### Loop ophalen

In [ ]:
verzamelde_data = []

# Loop over je de ingelezen CSV (met 'id' als index)
for id_csv, row in datapunten.iterrows():
    object_id_gbs = row['object-id_gbs']
    func_nr = row['func_dp_nr']
    naam = row['func_dp_naam']
    systeem = row['func_systeem']
    
    print(f"Ophalen data voor ID {id_csv}: {naam}...")
    
    # Haal ruwe data op via API
    samples = haal_meetwaarden_op(object_id_gbs, START_DATUM, EIND_DATUM, headers)
    
    # Voeg jouw extra csv informatie toe aan elk datapunt (Maken van het Flat-bestand)
    for sample in samples:
        verzamelde_data.append({
            'timestamp': sample.get('sampleTime'),
            'id': id_csv,
            'func_dp_nr': func_nr,
            'datapunt_naam': naam,
            'datapunt_systeem': systeem,
            'waarde': sample.get('value')
        })

# Maak het Long-DataFrame
df_long = pd.DataFrame(verzamelde_data)
# Zorg dat pandas snapt dat kolom 'timestamp' echt een datum/tijd is
df_long['timestamp'] = pd.to_datetime(df_long['timestamp'])

print(f"\nKlaar! {len(df_long)} metingen opgehaald.")
display(df_long.head())

#### Zet alle waarden om naar numerieke waarden

Sommige statussen komen binnen als `offonEnumSet.0` of `offonEnumSet.1`. Deze zetten we om naar $0$ of $1$.

In [ ]:
# bewaar origineel
if 'waarde_raw' not in df_long.columns:
    df_long['waarde_raw'] = df_long['waarde']

# normaliseer en parseer
vals_clean = df_long['waarde_raw'].astype(str).str.strip().str.replace(',', '.', regex=False)
parsed = pd.to_numeric(vals_clean.str.rstrip('%'), errors='coerce')

# voor resterende niet-parseerbare strings: haal laatste cijfer
mask_missing = parsed.isna()
extracted = vals_clean.str.extract(r'(\d+)$')[0]
parsed.loc[mask_missing] = pd.to_numeric(extracted.loc[mask_missing], errors='coerce')

# zet als finale numerieke kolom terug (Int64 indien geheel)
df_long['waarde'] = parsed
if df_long['waarde'].dropna().apply(float.is_integer).all():
    df_long['waarde'] = df_long['waarde'].astype('Int64')

# checks
print("Totaal:", len(df_long), "Niet-NaN:", df_long['waarde'].notna().sum(), "NaN:", df_long['waarde'].isna().sum())

In [ ]:
print(df_long.head())

### Maak de aggregaties

In [ ]:
from aggregatie_dict import AGG_REGELS

In [ ]:
# Map aggregatieregels (default 'mean' als niet gespecificeerd)
df_long['agg_methode'] = df_long['func_dp_nr'].map(AGG_REGELS).fillna('mean')

# Voer per aggregatiemethode grouping & aggregatie uit
parts = []
group_cols = ['timestamp', 'func_dp_nr', 'datapunt_naam', 'datapunt_systeem']
for methode in df_long['agg_methode'].unique():
    sub = df_long[df_long['agg_methode'] == methode]
    agg = sub.groupby(group_cols, as_index=False)['waarde'].agg(methode)
    parts.append(agg)

df_clean = pd.concat(parts, ignore_index=True).sort_values(by=['timestamp','func_dp_nr']).reset_index(drop=True)
display(df_clean.head())

#### Maak een tabel voor het trainen van het model

In [ ]:
df_wide = (
    df_long
    .groupby(['timestamp', 'func_dp_nr'], as_index=False)['waarde']
    .mean()                                   # aggregeer dubbele metingen
    .pivot(index='timestamp', columns='func_dp_nr', values='waarde')
)

# verwijder de kolom-indexnaam en zet timestamp terug als kolom
df_wide.columns.name = None
df_wide = df_wide.reset_index().sort_values('timestamp').reset_index(drop=True)

# (optioneel) prefix feature-kolommen: f_1, f_2, ...
new_names = {}
for c in df_wide.columns:
    if c == 'timestamp':
        continue
    try:
        new_names[c] = f"f_{int(c)}"
    except Exception:
        new_names[c] = f"f_{c}"
df_wide = df_wide.rename(columns=new_names)
df_wide = df_wide.set_index('timestamp')

display(df_wide.head())
print("shape:", df_wide.shape)

### Dedecteer cumul en bereken diff

In [ ]:
# Itereren over alle feature-kolommen in de brede dataset
for col in df_wide.columns:
    # Bereken het verschil met de vorige tijdstap (huidige waarde - vorige waarde)
    diffs = df_wide[col].diff()
    
    # Filter de NaN-waarden en de momenten waarop er geen verandering was (verschil == 0)
    veranderingen = diffs.dropna()
    actieve_veranderingen = veranderingen[veranderingen != 0]
    
    # Check of de variabele cumulatief is:
    # 1. Er moeten veranderingen zijn.
    # 2. Vrijwel alle (> 99%) actieve veranderingen moeten positief zijn (stijgende tellerstand).
    if len(actieve_veranderingen) > 0 and (actieve_veranderingen > 0).mean() >= 0.99:
        print(f"Cumulatieve trend gedetecteerd in {col}. Wordt omgezet naar periodiek verbruik.")
        
        # Vervang de kolom door het berekende verschil.
        # .clip(lower=0) is een robuuste veiligheidsmaatregel voor 'meter resets' 
        # (bijv. teller gaat van 9999 naar 0, wat een enorme negatieve piek zou geven).
        df_wide[col] = diffs.clip(lower=0)

# De eerste rij zal altijd NaN zijn na een .diff(), deze kunnen we opvullen met 0 of de rij weglaten
df_wide = df_wide.fillna(0)

## STAP 5 - Weerdata toevoegen

Website -> https://opendata.meteo.be/download  

> **Product**: `Automatic weather station (AWS)`  
> **Layer**: `aws_10min`  
> **Format**: `CSV`  

In [ ]:
# Laad de csv in
weer = pd.read_csv("raw/aws_10min.csv")

# Selecteer enkel de data voor het punt van gent
weer = weer[weer['the_geom'] == 'POINT (50.98 3.816)']

# Verwijder kolommen die niet nodig zijn
col_to_keep = ['timestamp', 'temp_dry_shelter_avg', 'humidity_rel_shelter_avg', 'wind_speed_10m', 'wind_direction', 'sun_int_avg']

weer = weer[col_to_keep]

weer.head()

### Voeg de kolommen toe als f_51, f_52, f_53, f_54, f_55

In [ ]:
# Zorg timestamp kolom / index consistent
df_wide = df_wide.reset_index() if 'timestamp' not in df_wide.columns else df_wide.copy()
df_wide['timestamp'] = pd.to_datetime(df_wide['timestamp'], utc=True)

weer['timestamp'] = pd.to_datetime(weer['timestamp'], utc=True)

# (optioneel) afronden op 10-minuten als nodig:
# df_wide['timestamp'] = df_wide['timestamp'].dt.round('10T')
# weer['timestamp'] = weer['timestamp'].dt.round('10T')

# Selecteer & hernoem weer-kolommen naar features f_51..f_55
map_cols = {
    'temp_dry_shelter_avg': 'f_51',
    'humidity_rel_shelter_avg': 'f_52',
    'wind_speed_10m': 'f_53',
    'wind_direction': 'f_54',
    'sun_int_avg': 'f_55'
}
weer_sub = weer[['timestamp'] + list(map_cols.keys())].drop_duplicates(subset='timestamp')
weer_sub = weer_sub.rename(columns=map_cols)

# Merge (left join zodat alle rijen van df_wide blijven)
df_merged = df_wide.merge(weer_sub, on='timestamp', how='left')

# Controle: hoeveel waarden ontbraken bij de merge?
missing_counts = df_merged[['f_51','f_52','f_53','f_54','f_55']].isna().sum()
print("Ontbrekende waarden per weer-feature:\n", missing_counts)

# Extra check: hoeveel timestamps in df_wide missen in weer en omgekeerd
df_t = pd.to_datetime(df_wide['timestamp'], utc=True).drop_duplicates()
weer_t = pd.to_datetime(weer_sub['timestamp'], utc=True).drop_duplicates()
print("timestamps in df_wide not in weer:", (~df_t.isin(weer_t)).sum())
print("timestamps in weer not in df_wide:", (~weer_t.isin(df_t)).sum())

# (optioneel) zet timestamp weer als index
df_merged = df_merged.sort_values('timestamp').reset_index(drop=True).set_index('timestamp')

# resultaat tonen
display(df_merged.head())

## Sla de dataset op voor verdere analyse

In [ ]:
df_merged.to_csv(f'preprocessed/{GEBOUWNAAM}.csv', index=True, date_format='%Y-%m-%dT%H:%M:%SZ')